
# Corrected Intervention / Faithfulness Validation for Biomedical LLM Attention Graphs

This notebook is a corrected version of the uploaded **BioGPT Dangling Centrality Intervention Validation** notebook.

It preserves the study's core definitions of:

- BioGPT multiple-choice scoring,
- original Dangling Centrality \(\phi_C\),
- normalized outgoing-attention entropy \(H_i\),
- refined entropy \(H_i^*=H_i(1-\widetilde d_i)\),
- proposed \(\Omega_{MH}(v_i)=\phi_C(v_i)H_i^*\).

The corrections specifically address the problems found in the pilot run:

1. **BioGPT subwords are merged into words before graph analysis.**
2. The graph uses the manuscript's **five strongest positive outgoing edges per word**, not an arbitrary attention threshold.
3. Rankings are restricted to **question-stem words before selection**.
4. The intervention removes **exactly the same number of question words** for every method.
5. Interventions use **5%, 10%, and 20%** of eligible question words.
6. Probability drop is measured for the **original predicted option**, even if the perturbed prediction flips.
7. The main comparison is \(\Omega_{MH}\), \(\phi_C\), raw attention, and random deletion.
8. Random deletion is repeated and averaged **within each question** before paired statistical testing.
9. Statistical comparison uses **paired Wilcoxon signed-rank tests**, appropriate because the same questions are evaluated under each ranking.
10. The notebook reports **AOPC**, flip rate, paired win rate, and bootstrap confidence intervals.

**Do not use the old pilot numbers in the manuscript.** Run this corrected notebook and use only the new results.


## 1. Install packages

In [1]:

!pip -q install "transformers>=4.57,<5" "datasets>=4.0,<5" accelerate sentencepiece sacremoses networkx scipy openpyxl


## 2. Imports and reproducibility

In [2]:

import gc
import json
import math
import random
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from scipy.stats import wilcoxon
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", DEVICE)


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Device: cuda


## 3. Configuration

In [3]:

MODEL_NAME = "microsoft/biogpt"
DATASET_NAME = "openlifescienceai/medmcqa"
DATASET_SPLIT = "validation"

# First run the pilot. After checking it, set RUN_FULL_200=True.
RUN_FULL_200 = True
PILOT_INDEX = 0
N_QUESTIONS = 200

MAX_INPUT_TOKENS = 256

# This matches the revised manuscript / earlier graph-construction notebook.
TOP_K_EDGES_PER_NODE = 5

# Faithfulness intervention sizes.
REMOVAL_FRACTIONS = [0.05, 0.10, 0.20]

# Random baseline repetitions per question and fraction.
RANDOM_REPEATS = 20

# Keep the same degree normalization used in the uploaded intervention notebook:
# normalized degree = degree(v) / max_degree(G)
EPS = 1e-12

# Manuscript baseline, used only as a consistency warning after the full run.
REPORTED_BASELINE_ACCURACY = 0.405
BASELINE_TOLERANCE = 0.02

OUTPUT_DIR = Path("/content/biogpt_faithfulness_corrected")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)


Output directory: /content/biogpt_faithfulness_corrected


## 4. Load MedMCQA

In [4]:

ds = load_dataset(
    DATASET_NAME,
    split=DATASET_SPLIT,
    trust_remote_code=True
)

print("Samples:", len(ds))
print("Columns:", ds.column_names)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openlifescienceai/medmcqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openlifescienceai/medmcqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Samples: 4183
Columns: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name']


### Normalize the MedMCQA schema

In [5]:

def normalize_medmcqa_row(row):
    q = row.get("question") or row.get("Question")
    options = []

    # Native/common MedMCQA schema.
    for key in ["opa", "opb", "opc", "opd"]:
        if key in row:
            options.append(str(row[key]).strip())

    # Fallback options schema.
    if not options and "options" in row:
        opt = row["options"]

        if isinstance(opt, dict):
            if "key" in opt and "value" in opt:
                ordered = list(zip(opt["key"], opt["value"]))
                options = [str(v).strip() for _, v in ordered]
            else:
                options = [str(opt[k]).strip() for k in sorted(opt)]
        else:
            options = list(map(lambda x: str(x).strip(), opt))

    if len(options) != 4:
        raise ValueError(
            f"Could not recover four options. Keys: {list(row.keys())}"
        )

    gold = row.get("cop", row.get("answer_idx", row.get("answer", row.get("label"))))

    if isinstance(gold, str) and gold.strip().upper() in ["A", "B", "C", "D"]:
        gold_idx = "ABCD".index(gold.strip().upper())
    else:
        gold_idx = int(gold)

    return {
        "question": str(q).strip(),
        "options": options,
        "gold_idx": gold_idx
    }

example = normalize_medmcqa_row(ds[0])
example


{'question': 'Which of the following is not true for myelinated nerve fibers:',
 'options': ['Impulse through myelinated fibers is slower than non-myelinated fibers',
  'Membrane currents are generated at nodes of Ranvier',
  'Saltatory conduction of impulses is seen',
  'Local anesthesia is effective only when the nerve is not covered by myelin sheath'],
 'gold_idx': 0}

In [6]:
%pip install -q --upgrade sacremoses sentencepiece "transformers>=4.57,<5"

In [7]:
import sacremoses
from importlib.metadata import version

print("sacremoses:", version("sacremoses"))
print("Installed successfully")

sacremoses: 0.2.0
Installed successfully


In [9]:
%pip uninstall -y sacremoses
%pip install --no-cache-dir sacremoses

Found existing installation: sacremoses 0.2.0
Uninstalling sacremoses-0.2.0:
  Successfully uninstalled sacremoses-0.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 21.3 MB/s eta 0:00:00


## 5. Load BioGPT

In [8]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)

model.config.output_attentions = True
model.config.use_cache = False
model.config.return_dict = True
model.to(DEVICE)
model.eval()

print("Model:", model.__class__.__name__)
print("Layers:", model.config.num_hidden_layers)
print("Heads:", model.config.num_attention_heads)
print("Hidden size:", model.config.hidden_size)


`torch_dtype` is deprecated! Use `dtype` instead!


Model: BioGptForCausalLM
Layers: 24
Heads: 16
Hidden size: 1024


## 6. Multiple-choice scoring

In [9]:

def build_prompt(question, options):
    return (
        f"Question: {question}\n"
        f"A. {options[0]}\n"
        f"B. {options[1]}\n"
        f"C. {options[2]}\n"
        f"D. {options[3]}\n"
        "Answer:"
    )


def encode_prompt(question, options):
    return tokenizer(
        build_prompt(question, options),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        add_special_tokens=True,
    ).input_ids[0].cpu()


@torch.no_grad()
def continuation_logprob_from_ids(prompt_ids_1d, answer_text):
    continuation_ids = tokenizer(
        " " + str(answer_text),
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids[0].cpu()

    full = torch.cat([prompt_ids_1d.cpu(), continuation_ids], dim=0)
    prompt_len = int(prompt_ids_1d.shape[0])

    if full.shape[0] <= prompt_len:
        return -1e9

    full_gpu = full.unsqueeze(0).to(DEVICE)

    out = model(
        input_ids=full_gpu,
        use_cache=False,
        return_dict=True
    )

    logits = out.logits[:, :-1, :]
    labels = full_gpu[:, 1:]

    start = max(prompt_len - 1, 0)
    logits = logits[:, start:, :]
    labels = labels[:, start:]

    logp = torch.log_softmax(logits.float(), dim=-1)
    token_lp = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    return float(token_lp.mean().item())


@torch.no_grad()
def score_mcq_from_prompt_ids(prompt_ids_1d, options):
    # Preserve the uploaded notebook's option-continuation format.
    continuations = [
        f"{'ABCD'[i]}. {options[i]}"
        for i in range(4)
    ]

    scores = np.array(
        [
            continuation_logprob_from_ids(prompt_ids_1d, continuation)
            for continuation in continuations
        ],
        dtype=float
    )

    probs = np.exp(scores - scores.max())
    probs = probs / probs.sum()

    pred = int(np.argmax(probs))

    uncertainty = float(
        -(probs * np.log(probs + EPS)).sum() / np.log(len(probs))
    )

    return {
        "pred_idx": pred,
        "probs": probs,
        "pred_prob": float(probs[pred]),
        "uncertainty": uncertainty,
        "scores": scores,
    }


def score_mcq(question, options):
    return score_mcq_from_prompt_ids(
        encode_prompt(question, options),
        options
    )



## 7. Extract aggregated BioGPT attention

As in the uploaded notebook, attention is extracted from:

**question + four answer options + BioGPT's predicted answer**

Attention is averaged across all layers and heads. The next section then merges BioGPT subwords into words.


In [10]:

@torch.no_grad()
def extract_aggregated_attention(question, options, pred_idx):
    prompt = build_prompt(question, options)
    text = prompt + " " + f"{'ABCD'[pred_idx]}. {options[pred_idx]}"

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        add_special_tokens=True,
    )

    input_ids = enc.input_ids.to(DEVICE)

    out = model(
        input_ids=input_ids,
        output_attentions=True,
        use_cache=False,
        return_dict=True,
    )

    if out.attentions is None:
        raise RuntimeError("BioGPT returned no attention tensors.")

    # layer tensor: [batch, heads, seq, seq]
    stack = torch.stack(
        [layer_attention[0].float().cpu() for layer_attention in out.attentions],
        dim=0
    )

    # Mean across layers and heads.
    attn = stack.mean(dim=(0, 1)).numpy().astype(np.float64)

    ids_cpu = input_ids[0].cpu()
    tokens = tokenizer.convert_ids_to_tokens(ids_cpu.tolist())

    return {
        "text": text,
        "prompt": prompt,
        "input_ids": ids_cpu,
        "subword_tokens": tokens,
        "subword_attention": attn,
    }


## 8. Merge BioGPT subwords into words

In [11]:

SPECIAL_TOKENS = set(tokenizer.all_special_tokens)


def merge_biogpt_subwords(tokens):
    merged_tokens = []
    groups = []
    current_pieces = []
    current_indices = []

    def close_current():
        nonlocal current_pieces, current_indices

        if current_indices:
            text = "".join(current_pieces).strip()

            if not text:
                text = "".join(str(tokens[i]) for i in current_indices)

            merged_tokens.append(text)
            groups.append(current_indices.copy())

        current_pieces = []
        current_indices = []

    for index, raw_token in enumerate(tokens):
        token = str(raw_token)

        if token in SPECIAL_TOKENS:
            close_current()
            merged_tokens.append(token)
            groups.append([index])

        elif token.endswith("</w>"):
            current_pieces.append(token[:-4])
            current_indices.append(index)
            close_current()

        elif token.startswith("Ġ") or token.startswith("▁"):
            close_current()
            current_pieces = [token[1:]]
            current_indices = [index]

        elif token.startswith("##"):
            current_pieces.append(token[2:])
            current_indices.append(index)

        else:
            current_pieces.append(token)
            current_indices.append(index)

    close_current()

    flattened = [i for group in groups for i in group]

    if sorted(flattened) != list(range(len(tokens))):
        raise RuntimeError("Subword merging lost or duplicated indices.")

    return merged_tokens, groups


def aggregate_attention_to_words(subword_attention, groups):
    word_count = len(groups)
    word_attention = np.zeros(
        (word_count, word_count),
        dtype=np.float64
    )

    for source_word, source_indices in enumerate(groups):
        for target_word, target_indices in enumerate(groups):
            block = subword_attention[
                np.ix_(source_indices, target_indices)
            ]

            word_attention[source_word, target_word] = float(
                block.mean()
            )

    return word_attention


def merge_attention_bundle(bundle):
    merged_tokens, groups = merge_biogpt_subwords(
        bundle["subword_tokens"]
    )

    merged_attention = aggregate_attention_to_words(
        bundle["subword_attention"],
        groups
    )

    bundle = dict(bundle)
    bundle["merged_tokens"] = merged_tokens
    bundle["groups"] = groups
    bundle["merged_attention"] = merged_attention

    return bundle


## 9. Identify question-stem words before ranking

In [12]:

def find_subsequence(sequence, subsequence):
    if len(subsequence) == 0 or len(subsequence) > len(sequence):
        return None

    for start in range(len(sequence) - len(subsequence) + 1):
        if sequence[start:start + len(subsequence)] == subsequence:
            return start, start + len(subsequence)

    return None


def question_subword_span(full_input_ids, question):
    sequence = full_input_ids.tolist()

    question_ids = tokenizer(
        str(question).strip(),
        add_special_tokens=False,
    ).input_ids

    match = find_subsequence(sequence, question_ids)

    if match is None:
        raise RuntimeError(
            "Could not locate the question token sequence inside the "
            "attention input. Do not continue with this question."
        )

    return match


def keep_merged_word(token):
    token = str(token).strip()

    if token in SPECIAL_TOKENS:
        return False

    if not token:
        return False

    # Exclude punctuation-only reconstructed words.
    if not any(ch.isalnum() for ch in token):
        return False

    return True


def eligible_question_word_nodes(bundle, question):
    q_start, q_end = question_subword_span(
        bundle["input_ids"],
        question
    )

    eligible = []

    for word_index, subword_indices in enumerate(bundle["groups"]):
        if not subword_indices:
            continue

        inside_question = (
            min(subword_indices) >= q_start
            and max(subword_indices) < q_end
        )

        if (
            inside_question
            and keep_merged_word(bundle["merged_tokens"][word_index])
        ):
            eligible.append(word_index)

    return eligible, (q_start, q_end)


## 10. Build the sparse directed attention graph

In [13]:

def build_attention_graph(merged_tokens, merged_attention):
    G = nx.DiGraph()

    valid_nodes = [
        i
        for i, token in enumerate(merged_tokens)
        if keep_merged_word(token)
    ]

    for node_id in valid_nodes:
        G.add_node(
            int(node_id),
            token=str(merged_tokens[node_id]),
            position=int(node_id),
        )

    for source_id in valid_nodes:
        candidates = []

        for target_id in valid_nodes:
            if source_id == target_id:
                continue

            weight = float(
                merged_attention[source_id, target_id]
            )

            if np.isfinite(weight) and weight > 0:
                candidates.append((target_id, weight))

        candidates.sort(
            key=lambda item: (-item[1], item[0])
        )

        # Exact manuscript construction: five strongest positive
        # outgoing edges per retained word.
        for target_id, weight in candidates[:TOP_K_EDGES_PER_NODE]:
            G.add_edge(
                int(source_id),
                int(target_id),
                weight=float(weight),
            )

    return G



## 11. Original Dangling Centrality \(\phi_C\)

The original metric is computed on the **undirected structural projection** using unweighted shortest-path hops, as in the manuscript.


In [14]:

def communication_score(G):
    U = nx.Graph()
    U.add_nodes_from(G.nodes())
    U.add_edges_from(
        (u, v)
        for u, v in G.edges()
        if u != v
    )

    if U.number_of_nodes() < 2:
        return 0.0

    score = 0.0

    for source, lengths in nx.all_pairs_shortest_path_length(U):
        for target, distance in lengths.items():
            if source < target and distance > 0:
                score += 1.0 / float(distance)

    return score


def dangling_centrality(G):
    base = communication_score(G)

    if base <= 0:
        return {
            node: 0.0
            for node in G.nodes()
        }

    out = {}

    for node in G.nodes():
        Gi = G.copy()

        incident = (
            list(Gi.in_edges(node))
            + list(Gi.out_edges(node))
        )

        Gi.remove_edges_from(incident)

        perturbed = communication_score(Gi)

        out[node] = (
            base - perturbed
        ) / base

    return out



## 12. Attention entropy and proposed \(\Omega_{MH}\)

The degree normalization below is intentionally retained from the **uploaded intervention notebook**:

\[
\widetilde d_i = \frac{d_i}{\max_j d_j}.
\]

This avoids silently changing the definition of \(\Omega_{MH}\) during faithfulness validation.


In [15]:

def attention_entropy_and_omega(G, phi):
    if G.number_of_nodes() == 0:
        return {}, {}, {}, {}

    degree = dict(G.degree())
    max_degree = max(degree.values()) if degree else 1

    H = {}
    H_star = {}
    degree_norm = {}
    omega = {}

    for node in G.nodes():
        weights = np.array(
            [
                float(G[node][target]["weight"])
                for target in G.successors(node)
                if float(G[node][target]["weight"]) > 0
            ],
            dtype=float,
        )

        if len(weights) <= 1 or weights.sum() <= 0:
            h = 0.0
        else:
            p = weights / weights.sum()
            h = float(
                -(p * np.log(p + EPS)).sum()
                / np.log(len(p))
            )

        d_norm = (
            degree[node] / max_degree
            if max_degree > 0
            else 0.0
        )

        h_star = h * (1.0 - d_norm)

        H[node] = h
        degree_norm[node] = d_norm
        H_star[node] = h_star
        omega[node] = float(
            phi.get(node, 0.0) * h_star
        )

    return H, H_star, degree_norm, omega


## 13. Raw-attention baseline

In [16]:

def raw_attention_received(merged_attention, graph_nodes):
    # Raw attention baseline = total attention received by a word
    # from all other merged words. Self-attention is removed.
    received_matrix = merged_attention.copy()
    np.fill_diagonal(received_matrix, 0.0)

    received = received_matrix.sum(axis=0)

    return {
        int(node): float(received[int(node)])
        for node in graph_nodes
    }


## 14. Exact question-only erasure intervention

In [17]:

def n_to_remove(n_eligible, fraction):
    if n_eligible <= 0:
        return 0

    return min(
        n_eligible,
        max(1, int(math.ceil(fraction * n_eligible)))
    )


def ranked_question_nodes(scores, eligible_nodes, fraction):
    eligible_nodes = list(map(int, eligible_nodes))

    ordered = sorted(
        eligible_nodes,
        key=lambda node: (
            -float(scores.get(node, -np.inf)),
            int(node)
        )
    )

    k = n_to_remove(
        len(eligible_nodes),
        fraction
    )

    return ordered[:k]


def random_question_nodes(eligible_nodes, fraction, rng):
    eligible_nodes = np.asarray(
        list(map(int, eligible_nodes)),
        dtype=int
    )

    k = n_to_remove(
        len(eligible_nodes),
        fraction
    )

    if k == 0:
        return []

    return sorted(
        rng.choice(
            eligible_nodes,
            size=k,
            replace=False
        ).astype(int).tolist()
    )


def remove_selected_word_groups_from_prompt(
    prompt_ids,
    word_groups,
    selected_word_nodes,
):
    remove_subword_indices = set()

    for word_node in selected_word_nodes:
        remove_subword_indices.update(
            word_groups[int(word_node)]
        )

    # Because selected nodes are question-only, this operation
    # cannot delete answer-option tokens.
    kept_ids = [
        token_id
        for index, token_id in enumerate(prompt_ids.tolist())
        if index not in remove_subword_indices
    ]

    if len(kept_ids) == 0:
        raise RuntimeError(
            "Intervention removed the entire prompt."
        )

    return torch.tensor(
        kept_ids,
        dtype=torch.long
    ), sorted(remove_subword_indices)


def run_intervention(
    prompt_ids,
    groups,
    options,
    base,
    selected_nodes,
):
    perturbed_prompt_ids, removed_subword_indices = (
        remove_selected_word_groups_from_prompt(
            prompt_ids,
            groups,
            selected_nodes,
        )
    )

    perturbed = score_mcq_from_prompt_ids(
        perturbed_prompt_ids,
        options
    )

    original_pred = int(base["pred_idx"])

    # Critical correction:
    # compare the probability of the ORIGINAL predicted option
    # before and after the intervention.
    original_prob_before = float(
        base["probs"][original_pred]
    )
    original_prob_after = float(
        perturbed["probs"][original_pred]
    )

    return {
        "pred_prob_drop": (
            original_prob_before
            - original_prob_after
        ),
        "prediction_flip": int(
            perturbed["pred_idx"] != original_pred
        ),
        "uncertainty_change": float(
            perturbed["uncertainty"]
            - base["uncertainty"]
        ),
        "perturbed_pred_idx": int(
            perturbed["pred_idx"]
        ),
        "original_choice_prob_after": (
            original_prob_after
        ),
        "removed_subword_count": len(
            removed_subword_indices
        ),
    }


## 15. Analyze one question

In [18]:

def analyze_question(row, qid=None, random_repeats=RANDOM_REPEATS):
    item = normalize_medmcqa_row(row)

    question = item["question"]
    options = item["options"]
    gold = item["gold_idx"]

    prompt_ids = encode_prompt(
        question,
        options
    )

    base = score_mcq_from_prompt_ids(
        prompt_ids,
        options
    )

    pred = int(base["pred_idx"])

    bundle = extract_aggregated_attention(
        question,
        options,
        pred
    )

    bundle = merge_attention_bundle(
        bundle
    )

    eligible_nodes, question_span = (
        eligible_question_word_nodes(
            bundle,
            question
        )
    )

    G = build_attention_graph(
        bundle["merged_tokens"],
        bundle["merged_attention"]
    )

    # Keep only question nodes that survived graph filtering.
    graph_nodes = set(G.nodes())
    eligible_nodes = [
        node
        for node in eligible_nodes
        if node in graph_nodes
    ]

    if len(eligible_nodes) < 3:
        raise RuntimeError(
            f"Only {len(eligible_nodes)} eligible question words remain."
        )

    phi = dangling_centrality(G)

    H, Hstar, degree_norm, omega = (
        attention_entropy_and_omega(
            G,
            phi
        )
    )

    raw_attention = raw_attention_received(
        bundle["merged_attention"],
        G.nodes()
    )

    methods = {
        "Omega_MH": omega,
        "phi_C": phi,
        "Raw_Attention": raw_attention,
    }

    record = {
        "qid": qid,
        "question": question,
        "gold_idx": gold,
        "pred_idx": pred,
        "correct": int(pred == gold),
        "base_pred_prob": float(
            base["pred_prob"]
        ),
        "base_uncertainty": float(
            base["uncertainty"]
        ),
        "n_graph_nodes": G.number_of_nodes(),
        "n_graph_edges": G.number_of_edges(),
        "n_eligible_question_words": len(
            eligible_nodes
        ),
        "max_phi": (
            max(
                phi[node]
                for node in eligible_nodes
            )
            if eligible_nodes
            else np.nan
        ),
        "max_omega": (
            max(
                omega[node]
                for node in eligible_nodes
            )
            if eligible_nodes
            else np.nan
        ),
    }

    intervention_rows = []

    for method_name, scores in methods.items():
        for fraction in REMOVAL_FRACTIONS:
            selected_nodes = ranked_question_nodes(
                scores,
                eligible_nodes,
                fraction
            )

            result = run_intervention(
                prompt_ids,
                bundle["groups"],
                options,
                base,
                selected_nodes
            )

            intervention_rows.append({
                "qid": qid,
                "method": method_name,
                "fraction": float(fraction),
                "repeat": 0,
                "n_eligible_question_words": len(
                    eligible_nodes
                ),
                "n_removed_words": len(
                    selected_nodes
                ),
                "removed_node_ids": json.dumps(
                    selected_nodes
                ),
                "removed_words": json.dumps(
                    [
                        bundle["merged_tokens"][node]
                        for node in selected_nodes
                    ],
                    ensure_ascii=False
                ),
                "original_pred_idx": pred,
                "gold_idx": gold,
                "original_correct": int(
                    pred == gold
                ),
                "original_pred_prob": float(
                    base["probs"][pred]
                ),
                **result,
            })

    # Random baseline: same eligible pool and exactly the same number
    # of removed words as every deterministic method.
    rng = np.random.default_rng(
        SEED + int(qid or 0) * 1000
    )

    for fraction in REMOVAL_FRACTIONS:
        for repeat in range(random_repeats):
            selected_nodes = random_question_nodes(
                eligible_nodes,
                fraction,
                rng
            )

            result = run_intervention(
                prompt_ids,
                bundle["groups"],
                options,
                base,
                selected_nodes
            )

            intervention_rows.append({
                "qid": qid,
                "method": "Random",
                "fraction": float(fraction),
                "repeat": int(repeat),
                "n_eligible_question_words": len(
                    eligible_nodes
                ),
                "n_removed_words": len(
                    selected_nodes
                ),
                "removed_node_ids": json.dumps(
                    selected_nodes
                ),
                "removed_words": json.dumps(
                    [
                        bundle["merged_tokens"][node]
                        for node in selected_nodes
                    ],
                    ensure_ascii=False
                ),
                "original_pred_idx": pred,
                "gold_idx": gold,
                "original_correct": int(
                    pred == gold
                ),
                "original_pred_prob": float(
                    base["probs"][pred]
                ),
                **result,
            })

    node_rows = []

    for node in eligible_nodes:
        node_rows.append({
            "qid": qid,
            "node_id": int(node),
            "word": bundle["merged_tokens"][node],
            "subword_indices": json.dumps(
                bundle["groups"][node]
            ),
            "phi_C": float(
                phi.get(node, np.nan)
            ),
            "H": float(
                H.get(node, np.nan)
            ),
            "degree_norm": float(
                degree_norm.get(node, np.nan)
            ),
            "H_star": float(
                Hstar.get(node, np.nan)
            ),
            "Omega_MH": float(
                omega.get(node, np.nan)
            ),
            "Raw_Attention": float(
                raw_attention.get(node, np.nan)
            ),
        })

    return {
        "record": record,
        "interventions": intervention_rows,
        "nodes": node_rows,
        "bundle": bundle,
        "eligible_nodes": eligible_nodes,
        "question_span": question_span,
        "base": base,
    }


## 16. Pilot validation — run this before the 200-question experiment

In [19]:

pilot = analyze_question(
    ds[PILOT_INDEX],
    qid=PILOT_INDEX,
    random_repeats=RANDOM_REPEATS
)

pilot_summary = pd.DataFrame(
    [pilot["record"]]
)

pilot_nodes = pd.DataFrame(
    pilot["nodes"]
)

pilot_interventions = pd.DataFrame(
    pilot["interventions"]
)

display(pilot_summary)

print("\nTop eligible QUESTION words by Omega_MH:")
display(
    pilot_nodes
    .sort_values(
        "Omega_MH",
        ascending=False
    )
    .head(15)
)

print("\nDeterministic intervention rows:")
display(
    pilot_interventions[
        pilot_interventions["method"] != "Random"
    ][[
        "method",
        "fraction",
        "n_eligible_question_words",
        "n_removed_words",
        "removed_words",
        "pred_prob_drop",
        "prediction_flip",
        "original_choice_prob_after",
    ]]
)


,qid,question,gold_idx,pred_idx,correct,base_pred_prob,base_uncertainty,n_graph_nodes,n_graph_edges,n_eligible_question_words,max_phi,max_omega
0,0,Which of the following is not true for myelina...,0,0,1,0.329789,0.983398,66,315,11,0.039898,0.015418



Top eligible QUESTION words by Omega_MH:


,qid,node_id,word,subword_indices,phi_C,H,degree_norm,H_star,Omega_MH,Raw_Attention
5,0,8,not,[9],0.024321,0.975270,0.35,0.633925,0.015418,0.521531
4,0,7,is,[8],0.024321,0.941612,0.35,0.612047,0.014886,0.625157
2,0,5,the,[6],0.023826,0.888224,0.30,0.621757,0.014814,0.740793
6,0,9,true,[10],0.026598,0.993090,0.45,0.546200,0.014528,0.527936
7,0,10,for,[11],0.029206,0.969030,0.50,0.484515,0.014151,0.848488
3,0,6,following,[7],0.026186,0.937127,0.45,0.515420,0.013497,0.676929
9,0,12,nerve,[13],0.030608,0.807870,0.50,0.403935,0.012364,0.682832
10,0,13,fibers,[14],0.031928,0.959281,0.60,0.383712,0.012251,0.712227
8,0,11,myelinated,[12],0.034618,0.934548,0.65,0.327092,0.011323,1.016648
1,0,4,of,[5],0.030938,0.687977,0.60,0.275191,0.008514,1.064547



Deterministic intervention rows:


,method,fraction,n_eligible_question_words,n_removed_words,removed_words,pred_prob_drop,prediction_flip,original_choice_prob_after
0,Omega_MH,0.05,11,1,"[""not""]",0.001891,0,0.327898
1,Omega_MH,0.10,11,2,"[""not"", ""is""]",0.018872,0,0.310917
2,Omega_MH,0.20,11,3,"[""not"", ""is"", ""the""]",0.013765,0,0.316023
3,phi_C,0.05,11,1,"[""Which""]",-0.018136,0,0.347924
4,phi_C,0.10,11,2,"[""Which"", ""myelinated""]",-0.032176,0,0.361965
5,phi_C,0.20,11,3,"[""Which"", ""myelinated"", ""fibers""]",-0.047470,0,0.377259
6,Raw_Attention,0.05,11,1,"[""Which""]",-0.018136,0,0.347924
7,Raw_Attention,0.10,11,2,"[""Which"", ""of""]",-0.012261,0,0.342050
8,Raw_Attention,0.20,11,3,"[""Which"", ""of"", ""myelinated""]",-0.027559,0,0.357348


In [20]:
import re

### Automatic pilot integrity checks

In [21]:

# 1. Every deterministic method must remove exactly the same count
#    at a given fraction.
count_check = (
    pilot_interventions[
        pilot_interventions["method"] != "Random"
    ]
    .groupby("fraction")["n_removed_words"]
    .nunique()
)

assert (count_check == 1).all(), (
    "Different methods removed different numbers of words."
)

# 2. Removed words must be non-empty for every deterministic row.
assert (
    pilot_interventions[
        pilot_interventions["method"] != "Random"
    ]["n_removed_words"] > 0
).all(), "A deterministic intervention removed zero words."

# 3. Eligible words are merged words, not raw tokenizer fragments.
fragment_markers = ("</w>", "##", "Ġ", "▁")

assert not pilot_nodes["word"].astype(str).str.contains(
    "|".join(map(re.escape, fragment_markers)),
    regex=True
).any(), (
    "Tokenizer fragment markers are still present after word merging."
)

# 4. Check expected removal counts from 5/10/20%.
n_eligible = int(
    pilot_summary.loc[
        0,
        "n_eligible_question_words"
    ]
)

expected_counts = {
    fraction: n_to_remove(
        n_eligible,
        fraction
    )
    for fraction in REMOVAL_FRACTIONS
}

observed_counts = (
    pilot_interventions[
        pilot_interventions["method"] == "Omega_MH"
    ]
    .set_index("fraction")[
        "n_removed_words"
    ]
    .to_dict()
)

assert observed_counts == expected_counts, (
    f"Removal-count mismatch: expected {expected_counts}, "
    f"observed {observed_counts}"
)

print("Pilot integrity checks PASSED.")
print("Eligible question words:", n_eligible)
print("Removal counts:", expected_counts)
print(
    "\nIf the displayed removed words are clinically/question-text words "
    "and the baseline prediction looks reasonable, set RUN_FULL_200=True."
)


Pilot integrity checks PASSED.
Eligible question words: 11
Removal counts: {0.05: 1, 0.1: 2, 0.2: 3}

If the displayed removed words are clinically/question-text words and the baseline prediction looks reasonable, set RUN_FULL_200=True.


## 17. Full 200-question experiment

In [22]:

summary_rows = []
intervention_rows = []
node_rows = []
failure_rows = []

if not RUN_FULL_200:
    print(
        "RUN_FULL_200 is False. "
        "After validating the pilot, set it to True in Section 3 "
        "and rerun the notebook."
    )

else:
    start_time = time.time()

    for qid in range(
        min(N_QUESTIONS, len(ds))
    ):
        try:
            result = analyze_question(
                ds[qid],
                qid=qid,
                random_repeats=RANDOM_REPEATS
            )

            summary_rows.append(
                result["record"]
            )

            intervention_rows.extend(
                result["interventions"]
            )

            node_rows.extend(
                result["nodes"]
            )

        except Exception as error:
            failure_rows.append({
                "qid": qid,
                "error": repr(error)
            })

        if (qid + 1) % 10 == 0:
            elapsed_minutes = (
                time.time() - start_time
            ) / 60.0

            print(
                f"[{qid + 1}/{min(N_QUESTIONS, len(ds))}] "
                f"elapsed={elapsed_minutes:.1f} min | "
                f"failures={len(failure_rows)}"
            )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    summary_df = pd.DataFrame(
        summary_rows
    )

    intervention_df = pd.DataFrame(
        intervention_rows
    )

    node_df = pd.DataFrame(
        node_rows
    )

    failure_df = pd.DataFrame(
        failure_rows
    )

    print("Completed questions:", len(summary_df))
    print("Failed questions:", len(failure_df))

    display(summary_df.head())
    display(failure_df.head())


[10/200] elapsed=0.9 min | failures=0
[20/200] elapsed=1.8 min | failures=0
[30/200] elapsed=2.7 min | failures=0
[40/200] elapsed=3.5 min | failures=0
[50/200] elapsed=4.4 min | failures=0
[60/200] elapsed=5.2 min | failures=0
[70/200] elapsed=6.1 min | failures=0
[80/200] elapsed=7.0 min | failures=0
[90/200] elapsed=7.8 min | failures=1
[100/200] elapsed=8.6 min | failures=1
[110/200] elapsed=9.5 min | failures=1
[120/200] elapsed=10.3 min | failures=1
[130/200] elapsed=11.2 min | failures=1
[140/200] elapsed=12.1 min | failures=1
[150/200] elapsed=12.9 min | failures=1
[160/200] elapsed=13.8 min | failures=1
[170/200] elapsed=14.6 min | failures=1
[180/200] elapsed=15.5 min | failures=1
[190/200] elapsed=16.4 min | failures=1
[200/200] elapsed=17.2 min | failures=1
Completed questions: 199
Failed questions: 1


,qid,question,gold_idx,pred_idx,correct,base_pred_prob,base_uncertainty,n_graph_nodes,n_graph_edges,n_eligible_question_words,max_phi,max_omega
0,0,Which of the following is not true for myelina...,0,0,1,0.329789,0.983398,66,315,11,0.039898,0.015418
1,1,Which of the following is not true about glome...,0,0,1,0.283704,0.997462,92,445,11,0.033462,0.013375
2,2,A 29 yrs old woman with a pregnancy of 17 week...,2,2,1,0.301514,0.973334,122,595,33,0.029157,0.010181
3,3,Axonal transport is:,2,0,0,0.497562,0.703024,17,70,3,0.123377,0.037662
4,4,Low insulin to glucagon ratio is seen in all o...,0,2,0,0.396463,0.876086,26,115,12,0.134869,0.046494


,qid,error
0,80,RuntimeError('Only 2 eligible question words r...


## 18. Baseline consistency check

In [23]:

if RUN_FULL_200 and len(summary_df):
    observed_accuracy = float(
        summary_df["correct"].mean()
    )

    observed_mean_probability = float(
        summary_df["base_pred_prob"].mean()
    )

    print(
        "Corrected-notebook baseline accuracy:",
        observed_accuracy
    )

    print(
        "Mean predicted-option probability:",
        observed_mean_probability
    )

    print(
        "Previously reported manuscript accuracy:",
        REPORTED_BASELINE_ACCURACY
    )

    difference = abs(
        observed_accuracy
        - REPORTED_BASELINE_ACCURACY
    )

    if difference > BASELINE_TOLERANCE:
        print(
            "\nWARNING: The baseline differs materially from the manuscript. "
            "Do not combine the new faithfulness table with the old 40.5% "
            "baseline until the scoring pipeline is reconciled."
        )
    else:
        print(
            "\nBaseline consistency check passed within tolerance."
        )


Corrected-notebook baseline accuracy: 0.39195979899497485
Mean predicted-option probability: 0.4785539885914422
Previously reported manuscript accuracy: 0.405

Baseline consistency check passed within tolerance.


## 19. Aggregate random repetitions within each question

In [24]:

def aggregate_random_repeats(intervention_df):
    deterministic = (
        intervention_df[
            intervention_df["method"] != "Random"
        ][[
            "qid",
            "method",
            "fraction",
            "pred_prob_drop",
            "prediction_flip",
            "uncertainty_change",
        ]]
        .copy()
    )

    random_avg = (
        intervention_df[
            intervention_df["method"] == "Random"
        ]
        .groupby(
            ["qid", "fraction"],
            as_index=False
        )
        .agg(
            pred_prob_drop=(
                "pred_prob_drop",
                "mean"
            ),
            prediction_flip=(
                "prediction_flip",
                "mean"
            ),
            uncertainty_change=(
                "uncertainty_change",
                "mean"
            ),
        )
    )

    random_avg["method"] = "Random"

    return pd.concat(
        [
            deterministic,
            random_avg[
                deterministic.columns
            ]
        ],
        ignore_index=True
    )


if RUN_FULL_200 and len(intervention_df):
    paired_df = aggregate_random_repeats(
        intervention_df
    )

    display(paired_df.head())


,qid,method,fraction,pred_prob_drop,prediction_flip,uncertainty_change
0,0,Omega_MH,0.05,0.001891,0.0,0.000574
1,0,Omega_MH,0.10,0.018872,0.0,0.005430
2,0,Omega_MH,0.20,0.013765,0.0,0.005706
3,0,phi_C,0.05,-0.018136,0.0,-0.010462
4,0,phi_C,0.10,-0.032176,0.0,-0.014183


## 20. Main faithfulness table

In [25]:

if RUN_FULL_200 and len(intervention_df):
    summary_table = (
        paired_df
        .groupby(
            ["method", "fraction"],
            as_index=False
        )
        .agg(
            mean_prob_drop=(
                "pred_prob_drop",
                "mean"
            ),
            median_prob_drop=(
                "pred_prob_drop",
                "median"
            ),
            std_prob_drop=(
                "pred_prob_drop",
                "std"
            ),
            prediction_flip_rate=(
                "prediction_flip",
                "mean"
            ),
            mean_uncertainty_change=(
                "uncertainty_change",
                "mean"
            ),
            n_questions=(
                "qid",
                "nunique"
            ),
        )
    )

    display(
        summary_table.sort_values(
            ["fraction", "method"]
        )
    )


,method,fraction,mean_prob_drop,median_prob_drop,std_prob_drop,prediction_flip_rate,mean_uncertainty_change,n_questions
0,Omega_MH,0.05,0.003857,0.002595,0.027865,0.075377,0.001341,199
3,Random,0.05,0.001565,0.000517,0.016719,0.053266,0.000819,199
6,Raw_Attention,0.05,0.004206,0.001320,0.030299,0.060302,0.001534,199
9,phi_C,0.05,0.005669,0.001510,0.028775,0.080402,0.002172,199
1,Omega_MH,0.10,0.004824,0.004896,0.031298,0.075377,0.001496,199
4,Random,0.10,0.003293,0.002226,0.017419,0.064322,0.002106,199
7,Raw_Attention,0.10,0.006069,0.003608,0.032882,0.080402,0.003132,199
10,phi_C,0.10,0.006240,0.002912,0.032803,0.100503,0.002717,199
2,Omega_MH,0.20,0.002131,0.002832,0.040633,0.070352,0.000891,199
5,Random,0.20,0.005142,0.003374,0.026763,0.077889,0.003036,199


## 21. AOPC

In [26]:

if RUN_FULL_200 and len(intervention_df):
    aopc_question = (
        paired_df
        .groupby(
            ["qid", "method"],
            as_index=False
        )
        .agg(
            AOPC=(
                "pred_prob_drop",
                "mean"
            ),
            mean_flip=(
                "prediction_flip",
                "mean"
            ),
        )
    )

    aopc_summary = (
        aopc_question
        .groupby(
            "method",
            as_index=False
        )
        .agg(
            mean_AOPC=(
                "AOPC",
                "mean"
            ),
            median_AOPC=(
                "AOPC",
                "median"
            ),
            std_AOPC=(
                "AOPC",
                "std"
            ),
            mean_flip=(
                "mean_flip",
                "mean"
            ),
            n_questions=(
                "qid",
                "nunique"
            ),
        )
        .sort_values(
            "mean_AOPC",
            ascending=False
        )
    )

    display(aopc_summary)


,method,mean_AOPC,median_AOPC,std_AOPC,mean_flip,n_questions
3,phi_C,0.006924,0.004848,0.031995,0.095477,199
2,Raw_Attention,0.004473,0.002195,0.031374,0.068677,199
0,Omega_MH,0.003604,0.003595,0.030955,0.073702,199
1,Random,0.003333,0.001877,0.019211,0.065159,199


## 22. Paired Wilcoxon tests + bootstrap confidence intervals

In [27]:

def bootstrap_mean_difference(
    differences,
    n_boot=5000,
    seed=SEED,
):
    differences = np.asarray(
        differences,
        dtype=float
    )

    differences = differences[
        np.isfinite(differences)
    ]

    if len(differences) == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)

    boot_means = np.empty(
        n_boot,
        dtype=float
    )

    for i in range(n_boot):
        sample = rng.choice(
            differences,
            size=len(differences),
            replace=True
        )

        boot_means[i] = sample.mean()

    return tuple(
        np.quantile(
            boot_means,
            [0.025, 0.975]
        )
    )


def paired_test(
    paired_df,
    method_a,
    method_b,
):
    rows = []

    for fraction in sorted(
        paired_df["fraction"].unique()
    ):
        a = (
            paired_df[
                (paired_df["method"] == method_a)
                & (
                    paired_df["fraction"]
                    == fraction
                )
            ][["qid", "pred_prob_drop"]]
            .rename(
                columns={
                    "pred_prob_drop": "A"
                }
            )
        )

        b = (
            paired_df[
                (paired_df["method"] == method_b)
                & (
                    paired_df["fraction"]
                    == fraction
                )
            ][["qid", "pred_prob_drop"]]
            .rename(
                columns={
                    "pred_prob_drop": "B"
                }
            )
        )

        merged = (
            a.merge(
                b,
                on="qid",
                how="inner"
            )
            .dropna()
        )

        differences = (
            merged["A"]
            - merged["B"]
        )

        if (
            len(merged) >= 2
            and not np.allclose(
                differences.values,
                0
            )
        ):
            stat, p_value = wilcoxon(
                merged["A"],
                merged["B"],
                alternative="greater",
                zero_method="wilcox"
            )
        else:
            stat, p_value = (
                np.nan,
                np.nan
            )

        ci_low, ci_high = (
            bootstrap_mean_difference(
                differences.values,
                seed=(
                    SEED
                    + int(
                        fraction * 1000
                    )
                )
            )
        )

        rows.append({
            "method_A": method_a,
            "method_B": method_b,
            "fraction": float(fraction),
            "n_pairs": len(merged),
            "mean_A": float(
                merged["A"].mean()
            ),
            "mean_B": float(
                merged["B"].mean()
            ),
            "mean_difference_A_minus_B": float(
                differences.mean()
            ),
            "paired_win_rate_A_gt_B": float(
                (differences > 0).mean()
            ),
            "bootstrap_95CI_low": (
                float(ci_low)
            ),
            "bootstrap_95CI_high": (
                float(ci_high)
            ),
            "wilcoxon_stat": stat,
            "p_value_one_sided_A_gt_B": (
                p_value
            ),
        })

    return pd.DataFrame(rows)


if RUN_FULL_200 and len(intervention_df):
    stats_df = pd.concat(
        [
            paired_test(
                paired_df,
                "Omega_MH",
                "Random"
            ),
            paired_test(
                paired_df,
                "Omega_MH",
                "Raw_Attention"
            ),
            paired_test(
                paired_df,
                "Omega_MH",
                "phi_C"
            ),
            paired_test(
                paired_df,
                "phi_C",
                "Random"
            ),
        ],
        ignore_index=True
    )

    display(stats_df)


,method_A,method_B,fraction,n_pairs,mean_A,mean_B,mean_difference_A_minus_B,paired_win_rate_A_gt_B,bootstrap_95CI_low,bootstrap_95CI_high,wilcoxon_stat,p_value_one_sided_A_gt_B
0,Omega_MH,Random,0.05,199,0.003857,0.001565,0.002292,0.532663,-0.001074,0.005504,10779.0,0.154069
1,Omega_MH,Random,0.10,199,0.004824,0.003293,0.001531,0.492462,-0.002109,0.005159,10447.0,0.270603
2,Omega_MH,Random,0.20,199,0.002131,0.005142,-0.003011,0.462312,-0.007482,0.001284,8703.0,0.937363
3,Omega_MH,Raw_Attention,0.05,199,0.003857,0.004206,-0.000349,0.502513,-0.005838,0.005042,9428.0,0.699629
4,Omega_MH,Raw_Attention,0.10,199,0.004824,0.006069,-0.001245,0.457286,-0.007468,0.004937,9120.0,0.817225
5,Omega_MH,Raw_Attention,0.20,199,0.002131,0.003144,-0.001013,0.482412,-0.007934,0.005709,9612.0,0.616163
6,Omega_MH,phi_C,0.05,199,0.003857,0.005669,-0.001812,0.512563,-0.006977,0.003260,9617.0,0.566656
7,Omega_MH,phi_C,0.10,199,0.004824,0.006240,-0.001417,0.502513,-0.007391,0.004361,9424.0,0.658637
8,Omega_MH,phi_C,0.20,199,0.002131,0.008863,-0.006731,0.407035,-0.014190,0.000543,8323.0,0.970758
9,phi_C,Random,0.05,199,0.005669,0.001565,0.004104,0.547739,0.000456,0.008093,10988.0,0.100965


## 23. Manuscript-ready compact table

In [28]:

if RUN_FULL_200 and len(intervention_df):
    manuscript_rows = []

    for method in [
        "Random",
        "Raw_Attention",
        "phi_C",
        "Omega_MH",
    ]:
        row = {
            "Method": method
        }

        for fraction in REMOVAL_FRACTIONS:
            subset = paired_df[
                (
                    paired_df["method"]
                    == method
                )
                & (
                    paired_df["fraction"]
                    == fraction
                )
            ]

            pct = int(
                round(
                    fraction * 100
                )
            )

            row[
                f"DeltaP@{pct}%"
            ] = float(
                subset[
                    "pred_prob_drop"
                ].mean()
            )

            row[
                f"Flip@{pct}%"
            ] = float(
                subset[
                    "prediction_flip"
                ].mean()
            )

        row["AOPC"] = float(
            aopc_question[
                aopc_question["method"]
                == method
            ]["AOPC"].mean()
        )

        manuscript_rows.append(
            row
        )

    manuscript_table = pd.DataFrame(
        manuscript_rows
    )

    display(
        manuscript_table.style.format({
            column: "{:.4f}"
            for column
            in manuscript_table.columns
            if column != "Method"
        })
    )


,Method,DeltaP@5%,Flip@5%,DeltaP@10%,Flip@10%,DeltaP@20%,Flip@20%,AOPC
0,Random,0.0016,0.0533,0.0033,0.0643,0.0051,0.0779,0.0033
1,Raw_Attention,0.0042,0.0603,0.0061,0.0804,0.0031,0.0653,0.0045
2,phi_C,0.0057,0.0804,0.0062,0.1005,0.0089,0.1055,0.0069
3,Omega_MH,0.0039,0.0754,0.0048,0.0754,0.0021,0.0704,0.0036


## 24. Publication-ready figures

In [ ]:

if RUN_FULL_200 and len(intervention_df):
    probability_plot = (
        paired_df
        .groupby(
            ["method", "fraction"],
            as_index=False
        )
        .agg(
            mean_prob_drop=(
                "pred_prob_drop",
                "mean"
            )
        )
    )

    plt.figure(
        figsize=(8, 5)
    )

    for method in [
        "Omega_MH",
        "phi_C",
        "Raw_Attention",
        "Random",
    ]:
        subset = probability_plot[
            probability_plot["method"]
            == method
        ].sort_values("fraction")

        plt.plot(
            subset["fraction"] * 100,
            subset["mean_prob_drop"],
            marker="o",
            label=method,
        )

    plt.xlabel(
        "Eligible question words removed (%)"
    )
    plt.ylabel(
        "Mean drop in probability of original prediction"
    )
    plt.title(
        "Intervention-based faithfulness"
    )
    plt.legend()
    plt.tight_layout()

    probability_path = (
        OUTPUT_DIR
        / "faithfulness_probability_drop.png"
    )

    plt.savefig(
        probability_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    flip_plot = (
        paired_df
        .groupby(
            ["method", "fraction"],
            as_index=False
        )
        .agg(
            prediction_flip_rate=(
                "prediction_flip",
                "mean"
            )
        )
    )

    plt.figure(
        figsize=(8, 5)
    )

    for method in [
        "Omega_MH",
        "phi_C",
        "Raw_Attention",
        "Random",
    ]:
        subset = flip_plot[
            flip_plot["method"]
            == method
        ].sort_values("fraction")

        plt.plot(
            subset["fraction"] * 100,
            subset[
                "prediction_flip_rate"
            ],
            marker="o",
            label=method,
        )

    plt.xlabel(
        "Eligible question words removed (%)"
    )
    plt.ylabel(
        "Prediction flip rate"
    )
    plt.title(
        "Prediction stability under targeted deletion"
    )
    plt.legend()
    plt.tight_layout()

    flip_path = (
        OUTPUT_DIR
        / "faithfulness_flip_rate.png"
    )

    plt.savefig(
        flip_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Saved:", probability_path)
    print("Saved:", flip_path)


## 25. Case-study table

In [29]:

if RUN_FULL_200 and len(node_df):
    case_ids = (
        summary_df
        .sort_values(
            "max_omega",
            ascending=False
        )
        .head(3)["qid"]
        .tolist()
    )

    case_nodes = (
        node_df[
            node_df["qid"].isin(
                case_ids
            )
        ]
        .sort_values(
            ["qid", "Omega_MH"],
            ascending=[True, False]
        )
        .groupby("qid")
        .head(10)
    )

    display(
        summary_df[
            summary_df["qid"].isin(
                case_ids
            )
        ]
    )

    display(case_nodes)


,qid,question,gold_idx,pred_idx,correct,base_pred_prob,base_uncertainty,n_graph_nodes,n_graph_edges,n_eligible_question_words,max_phi,max_omega
5,5,Concentration of tropicamide:,0,0,1,0.418141,0.908822,15,60,3,0.153535,0.053380
118,119,Shortest root of maxillary 1st molar,0,0,1,0.485143,0.644067,20,85,6,0.168894,0.051756
176,177,IV ultrashort acting barbiturate:,2,0,0,0.551910,0.794121,16,65,4,0.130199,0.057004


,qid,node_id,word,subword_indices,phi_C,H,degree_norm,H_star,Omega_MH,Raw_Attention
72,5,5,tropicamide,"[6, 7]",0.127273,0.908726,0.538462,0.419412,0.053380,0.397331
71,5,4,of,[5],0.153535,0.720950,0.846154,0.110915,0.017029,0.590136
70,5,3,Concentration,[4],0.121212,0.000000,0.461538,0.000000,0.000000,0.505580
1669,119,4,root,[6],0.094775,0.893603,0.388889,0.546091,0.051756,0.406491
1672,119,7,1st,[9],0.098420,0.889903,0.444444,0.494391,0.048658,0.234870
1671,119,6,maxillary,[8],0.102066,0.842731,0.500000,0.421366,0.043007,0.335863
1673,119,8,molar,[10],0.109356,0.961004,0.611111,0.373724,0.040869,0.309536
1668,119,3,Shortest,"[4, 5]",0.089915,0.000000,0.333333,0.000000,0.000000,0.321791
1670,119,5,of,[7],0.168894,0.796083,1.000000,0.000000,0.000000,0.689515
2371,177,5,acting,[8],0.112116,0.944245,0.461538,0.508439,0.057004,0.231549


## 26. Save all outputs

In [30]:

if RUN_FULL_200 and len(summary_df):
    summary_df.to_csv(
        OUTPUT_DIR
        / "biogpt_question_summary_corrected.csv",
        index=False
    )

    intervention_df.to_csv(
        OUTPUT_DIR
        / "biogpt_intervention_results_corrected.csv",
        index=False
    )

    node_df.to_csv(
        OUTPUT_DIR
        / "biogpt_question_word_scores_corrected.csv",
        index=False
    )

    paired_df.to_csv(
        OUTPUT_DIR
        / "biogpt_intervention_paired_corrected.csv",
        index=False
    )

    summary_table.to_csv(
        OUTPUT_DIR
        / "biogpt_faithfulness_summary_corrected.csv",
        index=False
    )

    aopc_summary.to_csv(
        OUTPUT_DIR
        / "biogpt_aopc_summary_corrected.csv",
        index=False
    )

    stats_df.to_csv(
        OUTPUT_DIR
        / "biogpt_paired_wilcoxon_corrected.csv",
        index=False
    )

    manuscript_table.to_csv(
        OUTPUT_DIR
        / "biogpt_manuscript_faithfulness_table.csv",
        index=False
    )

    if len(failure_df):
        failure_df.to_csv(
            OUTPUT_DIR
            / "biogpt_failed_questions.csv",
            index=False
        )

    excel_path = (
        OUTPUT_DIR
        / "BioGPT_Faithfulness_Corrected.xlsx"
    )

    with pd.ExcelWriter(
        excel_path,
        engine="openpyxl"
    ) as writer:
        summary_df.to_excel(
            writer,
            sheet_name="question_summary",
            index=False
        )

        paired_df.to_excel(
            writer,
            sheet_name="paired_results",
            index=False
        )

        summary_table.to_excel(
            writer,
            sheet_name="faithfulness_summary",
            index=False
        )

        aopc_summary.to_excel(
            writer,
            sheet_name="AOPC",
            index=False
        )

        stats_df.to_excel(
            writer,
            sheet_name="paired_tests",
            index=False
        )

        manuscript_table.to_excel(
            writer,
            sheet_name="manuscript_table",
            index=False
        )

    archive_path = shutil.make_archive(
        "/content/BioGPT_Faithfulness_Corrected_Results",
        "zip",
        root_dir=OUTPUT_DIR
    )

    print("Excel:", excel_path)
    print("ZIP:", archive_path)

    print("\nSaved files:")
    for path in sorted(
        OUTPUT_DIR.iterdir()
    ):
        print(" -", path.name)


Excel: /content/biogpt_faithfulness_corrected/BioGPT_Faithfulness_Corrected.xlsx
ZIP: /content/BioGPT_Faithfulness_Corrected_Results.zip

Saved files:
 - BioGPT_Faithfulness_Corrected.xlsx
 - biogpt_aopc_summary_corrected.csv
 - biogpt_failed_questions.csv
 - biogpt_faithfulness_summary_corrected.csv
 - biogpt_intervention_paired_corrected.csv
 - biogpt_intervention_results_corrected.csv
 - biogpt_manuscript_faithfulness_table.csv
 - biogpt_paired_wilcoxon_corrected.csv
 - biogpt_question_summary_corrected.csv
 - biogpt_question_word_scores_corrected.csv



## 27. Interpretation for the revised manuscript

The intervention supports faithfulness only if the observed data support it.

The key tests are:

\[
\Delta P_{\Omega_{MH}}
>
\Delta P_{\mathrm{Random}}
\]

and the stronger comparison

\[
\Delta P_{\Omega_{MH}}
>
\Delta P_{\mathrm{RawAttention}}.
\]

For each 5%, 10%, and 20% deletion level, report:

- mean probability drop for the original prediction,
- prediction-flip rate,
- paired Wilcoxon p-value,
- paired win rate,
- bootstrap 95% confidence interval for the paired mean difference.

Also report AOPC across 5%, 10%, and 20%.

If \(\Omega_{MH}\) does **not** outperform the controls, do not claim that it is a more faithful attribution method. The negative result would instead define the scope and limitations of the graph-based explanation framework.
